# 🤖 Notebook 3: Modern Topic Modeling
## BERTopic · Top2Vec · CTM · LLM Labeling · Multilingual — BSAN 6200

**What you'll learn:**
- Why Bag-of-Words topic models have fundamental limitations
- How **BERTopic** uses BERT embeddings → UMAP → HDBSCAN → c-TF-IDF
- How to use BERTopic's advanced features: dynamic topics, hierarchical topics, merging
- **Top2Vec** — jointly embedding documents and words
- **Contextualized Topic Model (CTM)** — combines BoW and BERT in one model
- **LLM-assisted topic labeling** — auto-name topics at scale
- **Multilingual BERTopic** — one model, 50+ languages

**Dataset:** 20 Newsgroups (4 categories)  
**Prerequisite:** Notebooks 1 & 2 (concepts build on each other)

## 0. Install & Imports

In [ ]:
# Core packages
!pip install bertopic top2vec sentence-transformers -q
!pip install contextualized-topic-models -q

# Optional for LLM labeling (requires OpenAI key)
# !pip install openai -q

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import spacy
from collections import Counter

# Sklearn
from sklearn.datasets import fetch_20newsgroups

# BERTopic
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

# Top2Vec
try:
    from top2vec import Top2Vec
    TOP2VEC_AVAILABLE = True
except ImportError:
    print('⚠️  Top2Vec not installed — pip install top2vec')
    TOP2VEC_AVAILABLE = False

# CTM
try:
    from contextualized_topic_models.models.ctm import CombinedTM
    from contextualized_topic_models.utils.data_preparation import (
        TopicModelDataPreparation
    )
    CTM_AVAILABLE = True
except ImportError:
    print('⚠️  CTM not installed — pip install contextualized-topic-models')
    CTM_AVAILABLE = False

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# NLTK
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import subprocess; subprocess.run(['python','-m','spacy','download','en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

print('✅ Imports ready')

---
## 1. Why BoW Models Have Fundamental Limits

Before jumping to modern methods, let's understand exactly what they solve.

In [ ]:
# ── Demonstrate BoW limitations ──────────────────────────────────
print('══ Bag-of-Words Limitation 1: Word Order Ignored ══')
print()
doc1 = 'dog bites man'
doc2 = 'man bites dog'
print(f'Doc 1: "{doc1}"')
print(f'Doc 2: "{doc2}"')
print('In BoW, both docs have identical feature vectors: {{dog:1, bites:1, man:1}}')
print('The model cannot distinguish them.')
print()

print('══ Bag-of-Words Limitation 2: Synonyms Are Invisible ══')
print()
docs = ['I drive my automobile to work', 'She parked her car in the lot']
print(f'Doc 1: "{docs[0]}"')
print(f'Doc 2: "{docs[1]}"')
print('automobile ≠ car in BoW vocabulary — they are treated as completely different words.')
print('LDA/NMF/LSA cannot know these documents are about the same topic.')
print()

print('══ Solution: Sentence Embeddings ══')
print()
print('BERT-based models map "automobile" and "car" to nearby vectors in 768-dim space.')
print('Semantic similarity is captured even for words that never appeared together.')

# Demonstrate with SentenceTransformer
model_demo = SentenceTransformer('all-MiniLM-L6-v2')
embs = model_demo.encode(['automobile trip', 'car journey', 'space exploration'])
from numpy.linalg import norm
def cos_sim(a, b): return np.dot(a, b) / (norm(a) * norm(b))

print()
print('Cosine similarity (BERT embeddings):')
print(f'  "automobile trip" vs "car journey": {cos_sim(embs[0], embs[1]):.3f}  (should be HIGH)')
print(f'  "automobile trip" vs "space exploration": {cos_sim(embs[0], embs[2]):.3f}  (should be LOW)')

## 2. Load Dataset

In [ ]:
CATEGORIES = ['sci.space', 'rec.sport.baseball',
              'talk.politics.guns', 'comp.graphics']

newsgroups = fetch_20newsgroups(
    subset='all',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

documents   = newsgroups.data
labels      = newsgroups.target
label_names = newsgroups.target_names

# Light cleaning — BERTopic needs readable sentences, NOT tokens
def clean_for_bert(text):
    """Minimal cleaning for embedding-based models.
    Unlike BoW preprocessing, we keep punctuation and stopwords.
    BERT uses full sentence context.
    """
    # Remove emails, URLs, excessive whitespace
    text = re.sub(r'\S+@\S+|http\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

docs_cleaned = [clean_for_bert(d) for d in documents]

# Remove very short docs
docs_cleaned = [d for d in docs_cleaned if len(d.split()) >= 10]
print(f'Documents after cleaning: {len(docs_cleaned):,}')
print(f'Example: {docs_cleaned[0][:200]}...')

---
## 3. BERTopic

BERTopic's modular pipeline:

```
Documents → [BERT embed] → [UMAP reduce] → [HDBSCAN cluster] → [c-TF-IDF represent]
```

Each step is independently swappable. Let's walk through it.

### 3.1 Understand the Pipeline Components

In [ ]:
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.vectorizers import ClassTfidfTransformer

# ── Component 1: Embedding model ─────────────────────────────────
# Maps each document to a 384-dim semantic vector
# 'all-MiniLM-L6-v2' = fast and good quality
# 'all-mpnet-base-v2' = better quality but slower
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model: all-MiniLM-L6-v2')
print(f'  Embedding dim: 384')

# ── Component 2: UMAP ─────────────────────────────────────────────
# Reduces 384-dim → 5-dim while preserving local neighborhood structure
# n_components=5 is the BERTopic default (NOT 2D — that's just for viz)
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,    # local neighborhood size
    min_dist=0.0,      # allow tight clusters
    metric='cosine',
    random_state=42
)
print('\nUMAP: 384-dim → 5-dim reduction')

# ── Component 3: HDBSCAN ──────────────────────────────────────────
# Density-based clustering — automatically finds number of clusters
# Documents that don't fit any cluster → assigned to topic -1 (noise)
hdbscan_model = HDBSCAN(
    min_cluster_size=10,  # minimum docs to form a topic
    min_samples=5,        # core point threshold
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
print('\nHDBSCAN: automatic cluster detection (no k needed)')
print('  → Outlier documents assigned to topic -1')

# ── Component 4: c-TF-IDF ──────────────────────────────────────────
# Class-level TF-IDF: treats each cluster as one big document
# Finds words that are UNIQUE to that cluster vs. the rest
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
print('\nc-TF-IDF: class-level keyword extraction per topic')
print('  → Finds words that DISCRIMINATE each topic from all others')

### 3.2 Train BERTopic

In [ ]:
# ── Assemble BERTopic model ───────────────────────────────────────
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    ctfidf_model=ctfidf_model,
    representation_model=KeyBERTInspired(),  # better keyword extraction
    verbose=True
)

# ── Fit and transform ─────────────────────────────────────────────
print('Training BERTopic... (embedding step takes ~2-3 min)')
topics, probs = topic_model.fit_transform(docs_cleaned)

# Quick summary
n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = sum(1 for t in topics if t == -1)
print(f'\n✅ BERTopic trained')
print(f'   Topics found:     {n_topics}')
print(f'   Outlier docs (-1): {n_outliers} ({100*n_outliers/len(topics):.1f}%)')

### 3.3 Inspect Topics

In [ ]:
# ── Topic info dataframe ──────────────────────────────────────────
topic_info = topic_model.get_topic_info()
print('Topic summary (excluding outliers):')
print(topic_info[topic_info['Topic'] != -1].head(10).to_string(index=False))

In [ ]:
# ── Top words per topic ───────────────────────────────────────────
print('Top words per topic:\n')
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    words = topic_model.get_topic(topic_id)
    word_str = '  |  '.join([f'{w} ({s:.3f})' for w, s in words[:8]])
    print(f'Topic {topic_id}: {word_str}')

### 3.4 Visualizations

In [ ]:
# ── 2D UMAP visualization of topic clusters ───────────────────────
# Re-run UMAP at 2D for visualization only
umap_2d = UMAP(n_components=2, n_neighbors=15, min_dist=0.0,
               metric='cosine', random_state=42)

# Get cached embeddings
embeddings = topic_model._extract_embeddings(docs_cleaned, method='document')
coords_2d = umap_2d.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(12, 8))
unique_topics = sorted(set(topics))
palette = plt.cm.tab20.colors

for topic_id in unique_topics:
    mask = np.array(topics) == topic_id
    color = 'lightgray' if topic_id == -1 else palette[topic_id % 20]
    label = 'Outliers (-1)' if topic_id == -1 else f'T{topic_id}'
    ax.scatter(coords_2d[mask, 0], coords_2d[mask, 1],
               c=color, label=label, alpha=0.4, s=8)

ax.set_title('BERTopic — Document Clusters in 2D UMAP Space', fontsize=14)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8, ncol=2)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
plt.tight_layout()
plt.show()

In [ ]:
# ── Intertopic distance map (BERTopic built-in) ───────────────────
# Each bubble = one topic; size = prevalence; spacing = semantic distance
# Similar to pyLDAvis left panel
fig = topic_model.visualize_topics()
fig.update_layout(title='BERTopic — Intertopic Distance Map')
fig.show()  # interactive in Jupyter
# fig.write_html('bertopic_distance_map.html')  # save as HTML

In [ ]:
# ── Top words barchart ────────────────────────────────────────────
fig = topic_model.visualize_barchart(top_n_topics=8, n_words=8)
fig.update_layout(title='BERTopic — Top Words per Topic')
fig.show()

### 3.5 Dynamic Topics — How Topics Change Over Time

In [ ]:
# ── Simulate timestamps — use document index as proxy for time ────
# In real usage: timestamps = list of datetime objects per document
n = len(docs_cleaned)
# Assign each doc to one of 4 quarters
timestamps = [f'Q{(i//(n//4))+1}' for i in range(n)]

print(f'Timestamp distribution:')
for q, cnt in sorted(Counter(timestamps).items()):
    print(f'  {q}: {cnt} docs')

# Compute topics over time
topics_over_time = topic_model.topics_over_time(
    docs_cleaned,
    timestamps,
    nr_bins=4,
    datetime_format=None
)
print(f'\ntopics_over_time shape: {topics_over_time.shape}')
print(topics_over_time.head(8).to_string(index=False))

In [ ]:
# ── Visualize topic frequency over time ───────────────────────────
top_topic_ids = (
    topic_info[topic_info['Topic'] != -1]
    .nlargest(6, 'Count')['Topic']
    .tolist()
)
fig = topic_model.visualize_topics_over_time(
    topics_over_time,
    topics=top_topic_ids,
    title='BERTopic — Topic Prevalence Over Time'
)
fig.show()

### 3.6 Hierarchical Topics — Merge Similar Clusters

In [ ]:
# ── Build topic hierarchy ─────────────────────────────────────────
hierarchical_topics = topic_model.hierarchical_topics(docs_cleaned)
print('Hierarchical topic merges:')
print(hierarchical_topics[['Parent_ID','Parent_Name','Topics','Distance']].head(12).to_string(index=False))

In [ ]:
# ── Visualize the dendrogram ──────────────────────────────────────
fig = topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics,
    title='BERTopic — Topic Hierarchy (Dendrogram)'
)
fig.show()

### 3.7 Reduce & Merge Topics

In [ ]:
# ── Automatically merge down to N topics ──────────────────────────
# Useful when HDBSCAN found too many fine-grained topics
original_n = len(set(topics)) - (1 if -1 in topics else 0)
target_n   = min(6, original_n)  # reduce to max 6

if original_n > target_n:
    topic_model_reduced = topic_model.reduce_topics(
        docs_cleaned, nr_topics=target_n
    )
    new_topics = topic_model_reduced.topics_
    new_n = len(set(new_topics)) - (1 if -1 in new_topics else 0)
    print(f'Reduced: {original_n} → {new_n} topics')
else:
    topic_model_reduced = topic_model
    print(f'Already at {original_n} topics — no reduction needed')

### 3.8 Topic Naming

In [ ]:
# ── Manual naming based on word inspection ────────────────────────
print('Top words per topic for naming:\n')
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(topic_id)[:6]]
    print(f'Topic {topic_id}: {", ".join(words)}')

print()
print('Review the words above and fill in your topic names:')

# Adjust based on YOUR actual output
TOPIC_NAMES = {}
for tid in sorted(set(topics)):
    if tid == -1:
        TOPIC_NAMES[tid] = 'Outliers'
    else:
        top_word = topic_model.get_topic(tid)[0][0] if topic_model.get_topic(tid) else 'unknown'
        TOPIC_NAMES[tid] = f'Topic {tid}: {top_word}'  # placeholder — update after inspection

print('\nAssigned names:')
for tid, name in TOPIC_NAMES.items():
    print(f'  Topic {tid} → {name}')

In [ ]:
# ── Optional: LLM-based topic labeling ────────────────────────────
# This requires an OpenAI API key. Uncomment and set your key to use.

# from bertopic.representation import OpenAI
# import openai
#
# LLM_PROMPT = """
# I have a topic that contains the following keywords: [KEYWORDS]
# Based on these keywords, create a short, descriptive topic name
# (3-5 words max) and a one-sentence description.
# Return only JSON: {"name": "...", "description": "..."}
# """
#
# llm_repr = OpenAI(
#     client=openai.OpenAI(api_key='YOUR_KEY_HERE'),
#     model='gpt-4o-mini',
#     prompt=LLM_PROMPT,
#     nr_docs=5,           # representative docs to include in prompt
#     delay_in_seconds=1   # rate limiting
# )
#
# topic_model_llm = BERTopic(
#     embedding_model=embedding_model,
#     representation_model=llm_repr
# )
# topics_llm, _ = topic_model_llm.fit_transform(docs_cleaned)
# print(topic_model_llm.get_topic_info()[['Topic','Name']].head(10))

print('LLM labeling code above — uncomment and add your OpenAI API key to use.')

---
## 4. Top2Vec

Top2Vec jointly embeds **documents and words** in the same semantic space.  
Topic words are the vocabulary words **nearest to each cluster centroid** — 
no c-TF-IDF step needed.

**Pipeline:**
```
Embed docs (Doc2Vec or BERT) → UMAP → HDBSCAN → centroid → nearest vocab words
```

In [ ]:
if TOP2VEC_AVAILABLE:
    print('Training Top2Vec...')
    # speed options: 'fast-learn', 'learn', 'deep-learn'
    # 'learn' = Doc2Vec (fast), 'deep-learn' = sentence-transformers (accurate)
    tv_model = Top2Vec(
        documents=docs_cleaned,
        embedding_model='doc2vec',  # use 'sentence-transformers' for better results
        speed='learn',
        workers=4,
        min_count=10            # min document frequency for vocabulary
    )

    print(f'✅ Top2Vec trained')
    print(f'   Topics found: {tv_model.get_num_topics()}')
else:
    print("""
Top2Vec not installed. Here's the usage:

from top2vec import Top2Vec

# Fast (Doc2Vec embedding):
model = Top2Vec(documents=docs, embedding_model='doc2vec', speed='learn')

# Better quality (BERT embedding):
model = Top2Vec(documents=docs, embedding_model='sentence-transformers')

# Results:
print(model.get_num_topics())
topic_words, word_scores, topic_nums = model.get_topics(5)
""")

In [ ]:
if TOP2VEC_AVAILABLE:
    # ── Get topic words ────────────────────────────────────────────────
    n_topics_tv = tv_model.get_num_topics()
    topic_words_tv, word_scores_tv, topic_nums_tv = tv_model.get_topics(n_topics_tv)

    print('Top2Vec Topics (top 8 words):')
    for i, (words, scores) in enumerate(zip(topic_words_tv, word_scores_tv)):
        word_str = '  |  '.join([f'{w}({s:.3f})' for w, s in zip(words[:8], scores[:8])])
        print(f'  Topic {i}: {word_str}')

In [ ]:
if TOP2VEC_AVAILABLE:
    # ── Search for topics by keywords ────────────────────────────────
    print('Semantic search: topics related to "orbit launch satellite"')
    try:
        words, word_scores, topic_scores, topic_nums = tv_model.search_topics(
            keywords=['orbit', 'launch', 'satellite'],
            num_topics=3
        )
        for i, (num, score) in enumerate(zip(topic_nums, topic_scores)):
            print(f'  Topic {num} (score: {score:.3f}): {", ".join(words[i][:6])}')
    except Exception as e:
        print(f'  (Search unavailable: {e})')

In [ ]:
if TOP2VEC_AVAILABLE:
    # ── Reduce topics if needed ────────────────────────────────────────
    if n_topics_tv > 6:
        tv_model.hierarchical_topic_reduction(num_topics=6)
        print(f'Reduced to 6 topics')
        topic_words_r, _, _ = tv_model.get_topics(6)
        for i, words in enumerate(topic_words_r):
            print(f'  Topic {i}: {", ".join(words[:8])}')

---
## 5. Contextualized Topic Model (CTM)

CTM combines:
- **BoW** input → interpretable word-level topics (like LDA)
- **BERT** sentence embeddings → semantic context (like BERTopic)

Both are fused via a **Variational Autoencoder** → latent topic distribution.

This gives you the interpretability of LDA with the semantic richness of BERT — 
especially useful for **short texts** and **multilingual** corpora.

In [ ]:
if CTM_AVAILABLE:
    # ── Prepare data for CTM ──────────────────────────────────────────
    # CTM needs both raw text (for BERT) and a vocabulary (for BoW reconstruction)
    # TopicModelDataPreparation handles this

    qt = TopicModelDataPreparation('sentence-transformers/all-MiniLM-L6-v2')
    training_dataset = qt.fit(text_for_contextual=docs_cleaned,
                               text_for_bow=docs_cleaned)

    print(f'Vocabulary size: {len(qt.vocab)}')
    print(f'BERT dim: 384')
    print(f'Training samples: {len(training_dataset)}')

    # ── Train CTM ─────────────────────────────────────────────────────
    ctm = CombinedTM(
        bow_size=len(qt.vocab),
        contextual_size=384,    # matches all-MiniLM-L6-v2 output dim
        n_components=4,         # number of topics
        num_epochs=20,
        batch_size=64,
        lr=2e-3
    )
    ctm.fit(training_dataset)
    print('\n✅ CTM trained')

else:
    print("""
CTM not installed. Usage pattern:

from contextualized_topic_models.models.ctm import CombinedTM
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation

# Prepare data (handles both BoW and BERT)
qt = TopicModelDataPreparation('all-MiniLM-L6-v2')
training_dataset = qt.fit(text_for_contextual=docs, text_for_bow=docs)

# Train
ctm = CombinedTM(
    bow_size=len(qt.vocab),
    contextual_size=384,
    n_components=10,
    num_epochs=20
)
ctm.fit(training_dataset)
""")

In [ ]:
if CTM_AVAILABLE:
    # ── Extract and display topics ─────────────────────────────────────
    topic_words_ctm = ctm.get_topic_lists(10)
    print('CTM Topics (top 10 words):')
    for i, words in enumerate(topic_words_ctm):
        print(f'  Topic {i}: {", ".join(words)}')

In [ ]:
if CTM_AVAILABLE:
    # ── Document-topic distributions ─────────────────────────────────
    test_dataset = qt.transform(text_for_contextual=docs_cleaned[:20],
                                 text_for_bow=docs_cleaned[:20])
    topic_predictions = ctm.get_doc_topic_distribution(test_dataset, n_samples=20)

    df_ctm = pd.DataFrame(
        topic_predictions,
        columns=[f'Topic {i}' for i in range(ctm.n_components)]
    )
    print('CTM document-topic distributions (first 5 docs):')
    print(df_ctm.head().round(3).to_string())

---
## 6. Multilingual BERTopic

Using a multilingual sentence encoder maps documents in different languages to the same semantic space.  
Topics cluster by **meaning**, not by language.

Supported model: `paraphrase-multilingual-MiniLM-L12-v2` (50+ languages)

In [ ]:
# ── Multilingual demo corpus ──────────────────────────────────────
# Same topic expressed in 3 languages — should cluster together

multilingual_docs = [
    # Space cluster — English, French, German
    "NASA launched a new rocket to orbit the moon.",
    "La NASA a lancé une fusée pour orbiter la lune.",
    "Die NASA startete eine Rakete in die Umlaufbahn des Mondes.",
    "The satellite entered lunar orbit successfully.",
    "Le satellite est entré en orbite lunaire avec succès.",
    "Der Satellit trat erfolgreich in eine Mondorbahn ein.",

    # Sports cluster — English, Spanish, Portuguese
    "The baseball team won the championship game.",
    "El equipo de béisbol ganó el campeonato.",
    "O time de beisebol ganhou o campeonato.",
    "The pitcher threw a perfect strikeout in the final inning.",
    "El lanzador logró un ponche perfecto en la última entrada.",
    "O arremessador fez um strikeout perfeito no último inning.",

    # Tech cluster — English, Italian, Dutch
    "The graphics processor renders images at 60 frames per second.",
    "Il processore grafico renderizza immagini a 60 fotogrammi al secondo.",
    "De grafische processor rendert afbeeldingen op 60 frames per seconde.",
    "Software engineers developed a new image compression algorithm.",
    "Gli ingegneri software hanno sviluppato un nuovo algoritmo di compressione.",
    "Software-ingenieurs ontwikkelden een nieuw beeldcompressie-algoritme.",
]

langs = (['EN','FR','DE']*2) + (['EN','ES','PT']*2) + (['EN','IT','NL']*2)
print(f'{len(multilingual_docs)} docs in {len(set(langs))} languages')
print('Languages:', sorted(set(langs)))

In [ ]:
# ── Train multilingual BERTopic ───────────────────────────────────
# Key: use a multilingual sentence encoder
ml_embedding_model = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2'
)

from hdbscan import HDBSCAN
from umap import UMAP

ml_topic_model = BERTopic(
    embedding_model=ml_embedding_model,
    umap_model=UMAP(
        n_components=2,     # only 18 docs — keep low dim
        n_neighbors=4,
        min_dist=0.0,
        random_state=42
    ),
    hdbscan_model=HDBSCAN(
        min_cluster_size=3, # tiny dataset
        min_samples=1,
        prediction_data=True
    ),
    verbose=False
)

ml_topics, _ = ml_topic_model.fit_transform(multilingual_docs)

print('Multilingual BERTopic results:')
for i, (doc, topic, lang) in enumerate(zip(multilingual_docs, ml_topics, langs)):
    print(f'  [{lang}] Topic {topic:2d} | {doc[:55]}...')

In [ ]:
# ── Check: did same-topic docs cluster together across languages? ──
print('Expected: Space (rows 0-5), Sports (6-11), Tech (12-17)')
print('Found topics:')
for group, expected in [('Space', range(0,6)), ('Sports', range(6,12)), ('Tech', range(12,18))]:
    group_topics = [ml_topics[i] for i in expected]
    dominant = Counter(group_topics).most_common(1)[0][0]
    agreement = group_topics.count(dominant) / len(group_topics)
    status = '✅' if agreement >= 0.5 else '⚠️'
    print(f'  {status} {group}: topics = {group_topics} → dominant T{dominant} ({agreement:.0%} agreement)')

print()
print('If agreement is high across languages → multilingual BERTopic works!')

---
## 7. Full Method Comparison

In [ ]:
comparison = pd.DataFrame([
    {
        'Method':         'LDA',
        'Choose k?':      'Manual',
        'Input':          'BoW counts',
        'Short Text?':    '✗ Poor',
        'Multilingual?':  '✗ No',
        'Speed':          '★★★★',
        'Interpretable?': '★★★★',
        'Best For':       'Long docs, large corpora',
    },
    {
        'Method':         'NMF',
        'Choose k?':      'Manual',
        'Input':          'TF-IDF',
        'Short Text?':    '✓ OK',
        'Multilingual?':  '✗ No',
        'Speed':          '★★★★★',
        'Interpretable?': '★★★★★',
        'Best For':       'News, clear topics',
    },
    {
        'Method':         'LSA',
        'Choose k?':      'Manual',
        'Input':          'TF-IDF',
        'Short Text?':    '✓ OK',
        'Multilingual?':  '✗ No',
        'Speed':          '★★★★★',
        'Interpretable?': '★★★',
        'Best For':       'Semantic search / IR',
    },
    {
        'Method':         'BERTopic',
        'Choose k?':      'Automatic',
        'Input':          'Raw text',
        'Short Text?':    '★ Best',
        'Multilingual?':  '✓ Yes',
        'Speed':          '★★★',
        'Interpretable?': '★★★★',
        'Best For':       'General purpose — recommended default',
    },
    {
        'Method':         'Top2Vec',
        'Choose k?':      'Automatic',
        'Input':          'Raw text',
        'Short Text?':    '★ Good',
        'Multilingual?':  '✓ Yes (BERT mode)',
        'Speed':          '★★★',
        'Interpretable?': '★★★★',
        'Best For':       'Simpler pipeline, semantic search',
    },
    {
        'Method':         'CTM',
        'Choose k?':      'Manual',
        'Input':          'Raw text + BoW',
        'Short Text?':    '★ Good',
        'Multilingual?':  '✓ Yes (LaBSE)',
        'Speed':          '★★',
        'Interpretable?': '★★★★',
        'Best For':       'Multilingual + LDA-style output',
    },
])

comparison

## 8. Practical Decision Guide

In [ ]:
def recommend_model(corpus_size, avg_doc_length, multilingual, metadata_available, need_auto_k):
    """
    Simple decision function for choosing a topic modeling approach.

    Parameters:
    -----------
    corpus_size       : int   — number of documents
    avg_doc_length    : int   — average word count per document
    multilingual      : bool  — does corpus contain multiple languages?
    metadata_available: bool  — date/author/category available per doc?
    need_auto_k       : bool  — don't want to choose k manually?
    """
    recs = []

    if multilingual:
        recs.append('✅ BERTopic (multilingual-MiniLM-L12-v2 embedding)')
        recs.append('✅ CTM (LaBSE embedding)')
        return recs

    if avg_doc_length < 20:
        recs.append('✅ BERTopic — handles short text via BERT context')
        recs.append('✅ GSDMM — one-topic-per-doc assumption')
        return recs

    if need_auto_k:
        recs.append('✅ BERTopic — HDBSCAN finds k automatically')
        recs.append('✅ Top2Vec — also automatic')
        return recs

    if metadata_available:
        recs.append('✅ STM (tomotopy) — covariate-aware topics')
        recs.append('✅ BERTopic + covariate post-analysis')
        return recs

    if corpus_size > 100_000:
        recs.append('✅ Online LDA — memory-efficient streaming')
        recs.append('✅ NMF — very fast on TF-IDF')
        return recs

    # Default
    recs.append('✅ NMF or LDA — good balance of speed and interpretability')
    recs.append('✅ BERTopic — if semantic accuracy is priority')
    return recs

# Test scenarios
scenarios = [
    {'corpus_size':5000,  'avg_doc_length':200, 'multilingual':False, 'metadata_available':True,  'need_auto_k':False},
    {'corpus_size':50000, 'avg_doc_length':15,  'multilingual':False, 'metadata_available':False, 'need_auto_k':True},
    {'corpus_size':3000,  'avg_doc_length':100, 'multilingual':True,  'metadata_available':False, 'need_auto_k':False},
    {'corpus_size':500000,'avg_doc_length':300, 'multilingual':False, 'metadata_available':False, 'need_auto_k':False},
]
labels_s = ['Long docs + metadata', 'Short tweets, no k', 'Multilingual corpus', 'Huge corpus']

for label, sc in zip(labels_s, scenarios):
    recs = recommend_model(**sc)
    print(f'Scenario: {label}')
    for r in recs:
        print(f'  {r}')
    print()

---
## 9. Summary

| Concept | Key Point |
|---|---|
| **BoW limitations** | Ignores word order, synonyms, short text context — fixed by embeddings |
| **BERTopic pipeline** | Embed → UMAP → HDBSCAN → c-TF-IDF. Each step swappable. |
| **Automatic k** | HDBSCAN in BERTopic / Top2Vec finds clusters without you choosing k |
| **Outliers (topic -1)** | Documents that don't fit any topic — inspect and decide whether to reassign |
| **Dynamic topics** | `topics_over_time()` — tracks prevalence; DTM (Notebook 2) tracks vocabulary change |
| **Top2Vec vs BERTopic** | Top2Vec = simpler; BERTopic = more control, more features |
| **CTM** | BoW + BERT in one model — best for multilingual + interpretable output |
| **LLM labeling** | Auto-name topics using GPT-4/Claude on the top-word list — scales to 100+ topics |

**You now have the complete topic modeling toolkit:**
- Notebook 1: LDA, NMF, LSA — the BoW classics
- Notebook 2: Online LDA, Guided LDA, GSDMM, STM, DTM, Author-Topic — advanced extensions
- Notebook 3: BERTopic, Top2Vec, CTM — embedding-based modern methods